In [1]:
import pandas as pd
import numpy as np
import time
import surprise
from surprise import Reader, Dataset, SVD, NMF, KNNBasic
from surprise.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import os
import kagglehub
import gc
import math

In [2]:
class AnimeLoader:
    def __init__(self, sample_size=100000):
        self.sample_size = sample_size
        
        # 1. Download Data
        print("Locating Anime Dataset...")
        try:
            self.data_dir = kagglehub.dataset_download("noiruuuu/anime-recommendations-database-vol2")
        except:
            self.data_dir = "/kaggle/input/anime-recommendations-database-vol2"
            
        # Note: In this specific dataset, the file is often named 'anime.csv' (singular), not 'animes.csv'.
        # We check both to be safe.
        possible_names = ['animes.csv', 'anime.csv']
        self.anime_path = None
        for name in possible_names:
            path = os.path.join(self.data_dir, name)
            if os.path.exists(path):
                self.anime_path = path
                break
        
        if self.anime_path is None:
            raise FileNotFoundError(f"Could not find anime csv in {self.data_dir}")

        self.rating_path = os.path.join(self.data_dir, 'ratings.csv')
        
        # 2. Load Content (Metadata)
        self.movies_df = self._load_animes()
        
        # 3. Load Ratings (Lazy/Sampled)
        self.ratings_df = self._load_ratings_lazy()
        
        # 4. Filter Metadata to match sampled ratings
        valid_items = set(self.ratings_df['item_id'].unique())
        self.movies_df = self.movies_df[self.movies_df['item_id'].isin(valid_items)].copy()
        print(f"Final filtered content: {len(self.movies_df)} anime titles.")
        
        gc.collect()

    def _load_animes(self):
        print("Loading Anime Metadata...")
        df = pd.read_csv(self.anime_path)
        
        # --- FIX: Correct Column Mapping ---
        # The dataset uses 'anime_id', not 'uid'
        df = df.rename(columns={
            'anime_id': 'item_id', 
            'title': 'title', 
            'genres': 'genres'  # Ensure this matches the CSV header
        })
        
        # Safety check: if 'genres' column is missing (some versions call it 'genre'), try to fix it
        if 'genres' not in df.columns and 'genre' in df.columns:
             df = df.rename(columns={'genre': 'genres'})
        
        # Fill NA genres
        df['genres'] = df['genres'].fillna('')
        
        # Process Genres (Comma separated -> Space separated for TF-IDF)
        df['genre_str'] = df['genres'].str.replace(',', ' ')
        
        # Verify columns exist before returning
        required_cols = ['item_id', 'title', 'genre_str']
        for col in required_cols:
            if col not in df.columns:
                raise KeyError(f"Column '{col}' not found. Available columns: {list(df.columns)}")

        return df[required_cols]

    def _load_ratings_lazy(self):
        print(f"Loading top {self.sample_size} valid ratings (excluding 0s)...")
        
        chunks = []
        total_loaded = 0
        chunksize = 50000
        
        with pd.read_csv(self.rating_path, chunksize=chunksize) as reader:
            for chunk in reader:
                # Standardize names: anime_id -> item_id
                chunk = chunk.rename(columns={'anime_id': 'item_id', 'score': 'rating', 'rating': 'rating'})
                
                # Filter '0' ratings (Watched but not rated)
                chunk = chunk[chunk['rating'] > 0]
                
                # Filter items: only keep ratings for items we have metadata for
                valid_ids = set(self.movies_df['item_id'])
                chunk = chunk[chunk['item_id'].isin(valid_ids)]
                
                chunks.append(chunk)
                total_loaded += len(chunk)
                
                if total_loaded >= self.sample_size:
                    break
        
        if not chunks:
            raise ValueError("No valid ratings found!")

        df = pd.concat(chunks)
        
        if len(df) > self.sample_size:
            df = df.iloc[:self.sample_size]
            
        # Optimize Types
        df['user_id'] = df['user_id'].astype('int32')
        df['item_id'] = df['item_id'].astype('int32')
        df['rating'] = df['rating'].astype('float32')
        
        print(f"Loaded {len(df)} ratings.")
        return df

    def get_surprise_train_test(self, test_size=0.2):
        print("Converting to Surprise Dataset...")
        reader = Reader(rating_scale=(1, 10))
        data = Dataset.load_from_df(self.ratings_df[['user_id', 'item_id', 'rating']], reader)
        return train_test_split(data, test_size=test_size, random_state=42)

    def get_content_data(self):
        return self.movies_df

In [3]:
# --- 2. Base Recommender & Wrappers ---
class BaseRecommender:
    def fit(self, trainset): raise NotImplementedError
    def test(self, testset): raise NotImplementedError

class CFRecommender(BaseRecommender):
    def __init__(self, algorithm_name='svd'):
        self.algo_name = algorithm_name
        if algorithm_name == 'svd': self.model = SVD()
        elif algorithm_name == 'nmf': self.model = NMF()
        else: raise ValueError("Unknown CF Algorithm")
            
    def fit(self, trainset): self.model.fit(trainset)
    def test(self, testset): return self.model.test(testset)
    def predict(self, uid, iid): return self.model.predict(uid, iid).est

# --- 3. Content-Based (Updated for 1-10 Scale) ---
class ContentBasedRecommender(BaseRecommender):
    def __init__(self, movies_df):
        self.movies_df = movies_df
        self.item_ids = movies_df['item_id'].values
        self.item_to_index = {iid: idx for idx, iid in enumerate(self.item_ids)}
        self.tfidf_matrix = None
        self.user_profiles = defaultdict(list)
        
    def fit(self, trainset):
        # 1. TF-IDF
        print("Building TF-IDF matrix...")
        tfidf = TfidfVectorizer(stop_words='english', dtype=np.float32)
        self.tfidf_matrix = tfidf.fit_transform(self.movies_df['genre_str'])
        
        # 2. User Profiles
        # Threshold updated to 6.0 because scale is 1-10
        print("Building user profiles (Likes >= 6.0)...")
        for uid, iid, rating in trainset.all_ratings():
            raw_uid = trainset.to_raw_uid(uid)
            raw_iid = trainset.to_raw_iid(iid)
            
            # If user rated it > 6, we consider it "liked" for content matching
            if rating >= 6.0:
                if raw_iid in self.item_to_index:
                    idx = self.item_to_index[raw_iid]
                    self.user_profiles[raw_uid].append(idx)
                    
    def predict_score(self, uid, iid):
        if iid not in self.item_to_index: return 0
        target_idx = self.item_to_index[iid]
        liked_indices = self.user_profiles.get(uid, [])
        
        if not liked_indices: return 0
        
        # Similarity Calculation
        target_vec = self.tfidf_matrix[target_idx]
        user_vecs = self.tfidf_matrix[liked_indices]
        similarities = cosine_similarity(target_vec, user_vecs)
        
        if similarities.shape[1] == 0: return 0
        
        max_sim = similarities.max()
        
        # SCALING FIX: 
        # Similarity is 0-1. We need output 1-10.
        # Formula: 1 + (sim * 9) -> Range [1, 10]
        predicted_rating = 1 + (max_sim * 9) 
        return predicted_rating

    def test(self, testset):
        predictions = []
        for uid, iid, true_r in testset:
            est = self.predict_score(uid, iid)
            predictions.append(surprise.prediction_algorithms.predictions.Prediction(
                uid, iid, true_r, est, details={'was_impossible': False}
            ))
        return predictions

In [4]:
# # --- 4. Hybrid Recommender ---
# class HybridRecommender(BaseRecommender):
#     def __init__(self, svd_model, content_model, alpha=0.5):
#         self.svd = svd_model
#         self.content = content_model
#         self.alpha = alpha 
        
#     def fit(self, trainset):
#         self.svd.fit(trainset)
#         self.content.fit(trainset)
        
#     def test(self, testset):
#         predictions = []
#         for uid, iid, true_r in testset:
#             final_score = self.predict(uid, iid)
#             predictions.append(surprise.prediction_algorithms.predictions.Prediction(
#                 uid, iid, true_r, final_score, details={'was_impossible': False}
#             ))
#         return predictions

#     def predict(self, uid, iid):
#         svd_score = self.svd.predict(uid, iid)
#         content_score = self.content.predict_score(uid, iid)
#         return (self.alpha * svd_score) + ((1 - self.alpha) * content_score)



# --- Hybrid Recommender (Weighted CF + Content) ---
class HybridRecommender(BaseRecommender):
    def __init__(self, cf_model, content_model, alpha=0.5):
        """
        Args:
            cf_model: Collaborative filtering model (e.g., CFRecommender)
            content_model: Content-based model
            alpha: Weight for CF model (1-alpha for content model)
        """
        self.cf = cf_model
        self.content = content_model
        self.alpha = alpha
        self.prediction_stats = {'cf_fail': 0, 'cb_fail': 0, 'both_fail': 0, 'success': 0}
        
    def fit(self, trainset):
        """Fit both models"""
        self.cf.fit(trainset)
        self.content.fit(trainset)
    
    def predict(self, uid, iid):
        """
        Make hybrid prediction
        Returns Prediction-like object with .est attribute
        """
        cf_score = None
        cb_score = None
        
        # Get CF prediction
        try:
            cf_pred = self.cf.predict(uid, iid)
            cf_score = cf_pred.est if hasattr(cf_pred, 'est') else cf_pred
        except Exception as e:
            self.prediction_stats['cf_fail'] += 1
        
        # Get Content-Based prediction
        try:
            cb_pred = self.content.predict(uid, iid)
            cb_score = cb_pred.est if hasattr(cb_pred, 'est') else cb_pred
        except Exception as e:
            self.prediction_stats['cb_fail'] += 1
        
        # Combine scores
        if cf_score is not None and cb_score is not None:
            # Both worked - weighted average
            final_score = (self.alpha * cf_score) + ((1 - self.alpha) * cb_score)
            self.prediction_stats['success'] += 1
        elif cf_score is not None:
            # Only CF worked
            final_score = cf_score
            self.prediction_stats['success'] += 1
        elif cb_score is not None:
            # Only CB worked
            final_score = cb_score
            self.prediction_stats['success'] += 1
        else:
            # Both failed - return default
            final_score = 2.5  # Middle of 1-5 scale
            self.prediction_stats['both_fail'] += 1
        
        # Return Prediction-like object
        class PredictionResult:
            def __init__(self, est):
                self.est = est
        
        return PredictionResult(final_score)
    
    def test(self, testset):
        """Test method for compatibility"""
        from surprise.prediction_algorithms.predictions import Prediction
        
        predictions = []
        for uid, iid, true_r in testset:
            pred = self.predict(uid, iid)
            est = pred.est if hasattr(pred, 'est') else pred
            
            predictions.append(Prediction(
                uid, iid, true_r, est, details={'was_impossible': False}
            ))
        return predictions
    
    def print_diagnostics(self):
        """Print prediction statistics"""
        total = sum(self.prediction_stats.values())
        if total > 0:
            print("\n=== Hybrid Model Diagnostics ===")
            print(f"Total predictions: {total}")
            print(f"Successful: {self.prediction_stats['success']} ({100*self.prediction_stats['success']/total:.1f}%)")
            print(f"CF failures: {self.prediction_stats['cf_fail']} ({100*self.prediction_stats['cf_fail']/total:.1f}%)")
            print(f"CB failures: {self.prediction_stats['cb_fail']} ({100*self.prediction_stats['cb_fail']/total:.1f}%)")
            print(f"Both failed: {self.prediction_stats['both_fail']} ({100*self.prediction_stats['both_fail']/total:.1f}%)")

In [5]:
# # # --- 5. Evaluator (Memory Safe & Scale Adapted) ---
# # class Evaluator:
# #     def evaluate_model(self, model, trainset, testset, name):
# #         print(f"Testing {name}...")
# #         start_fit = time.time()
# #         model.fit(trainset)
# #         fit_time = time.time() - start_fit
        
# #         testset.sort(key=lambda x: x[0])
# #         start_test = time.time()
        
# #         rmse_sse, count, sum_precision, sum_recall, n_users = 0, 0, 0, 0, 0
# #         current_uid, user_preds = None, []
        
# #         # THRESHOLD UPDATE: Used 7.0 as 'Good' because scale is 1-10
# #         def process_user_buffer(preds, k=10, threshold=7.0):
# #             if not preds: return 0, 0
# #             preds.sort(key=lambda x: x[0], reverse=True)
# #             n_rel = sum((true_r >= threshold) for (_, true_r) in preds)
# #             n_rec_k = sum((est >= threshold) for (est, _) in preds[:k])
# #             n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold)) 
# #                                   for (est, true_r) in preds[:k])
# #             prec = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
# #             rec = n_rel_and_rec_k / n_rel if n_rel != 0 else 0
# #             return prec, rec

# #         for uid, iid, true_r in testset:
# #             if hasattr(model, 'predict'):
# #                 p = model.predict(uid, iid)
# #                 est = p.est if hasattr(p, 'est') else p
# #             else:
# #                 est = model.predict_score(uid, iid)
            
# #             rmse_sse += (est - true_r) ** 2
# #             count += 1
            
# #             if uid != current_uid:
# #                 if current_uid is not None:
# #                     p, r = process_user_buffer(user_preds)
# #                     sum_precision += p
# #                     sum_recall += r
# #                     n_users += 1
# #                 current_uid = uid
# #                 user_preds = []
# #             user_preds.append((est, true_r))
            
# #         if current_uid is not None and user_preds:
# #             p, r = process_user_buffer(user_preds)
# #             sum_precision += p
# #             sum_recall += r
# #             n_users += 1
            
# #         inference_time = time.time() - start_test
# #         rmse = np.sqrt(rmse_sse / count) if count > 0 else 0
# #         avg_precision = sum_precision / n_users if n_users > 0 else 0
# #         avg_recall = sum_recall / n_users if n_users > 0 else 0
        
# #         return {
# #             "Algorithm": name,
# #             "RMSE": round(rmse, 4),
# #             "Precision@10": round(avg_precision, 4),
# #             "Recall@10": round(avg_recall, 4),
# #             "Inference Time (s)": round(inference_time, 4)
# #         }



# # --- 5. Evaluator (Corrected Metrics) ---
# class Evaluator:
#     def evaluate_model(self, model, trainset, testset, name):
#         print(f"Testing {name}...")
        
#         # Training phase
#         start_fit = time.time()
#         model.fit(trainset)
#         fit_time = time.time() - start_fit
        
#         # Sort test set by user ID for per-user metrics
#         testset.sort(key=lambda x: x[0])
        
#         # Inference phase - measure ONLY prediction time
#         start_test = time.time()
#         predictions = []
#         for uid, iid, true_r in testset:
#             if hasattr(model, 'predict'):
#                 p = model.predict(uid, iid)
#                 est = p.est if hasattr(p, 'est') else p
#             else:
#                 est = model.predict_score(uid, iid)
#             predictions.append((uid, iid, est, true_r))
#         inference_time = time.time() - start_test
        
#         # Calculate RMSE
#         rmse_sse = sum((est - true_r) ** 2 for (_, _, est, true_r) in predictions)
#         count = len(predictions)
#         rmse = np.sqrt(rmse_sse / count) if count > 0 else 0
        
#         # Calculate Precision@10 and Recall@10 per user
#         sum_precision, sum_recall, n_users = 0, 0, 0
#         current_uid = None
#         user_preds = []
        
#         def process_user_buffer(preds, k=10, threshold=7.0):
#             """
#             Calculate Precision@k and Recall@k for one user.
            
#             Precision@k: fraction of top-k recommendations that are relevant
#             Recall@k: fraction of all relevant items that appear in top-k
#             """
#             if not preds:
#                 return 0, 0
            
#             # Sort by estimated score (descending) to get top-k recommendations
#             preds.sort(key=lambda x: x[0], reverse=True)
            
#             # Get top-k predictions
#             top_k = preds[:k]
            
#             # Count relevant items in top-k (based on TRUE ratings)
#             n_relevant_in_top_k = sum((true_r >= threshold) for (est, true_r) in top_k)
            
#             # Count total relevant items for this user (across all their test items)
#             n_relevant_total = sum((true_r >= threshold) for (est, true_r) in preds)
            
#             # Precision@k = relevant items in top-k / k
#             prec = n_relevant_in_top_k / min(k, len(top_k)) if len(top_k) > 0 else 0
            
#             # Recall@k = relevant items in top-k / total relevant items
#             rec = n_relevant_in_top_k / n_relevant_total if n_relevant_total > 0 else 0
            
#             return prec, rec
        
#         # Process predictions per user
#         for uid, iid, est, true_r in predictions:
#             if uid != current_uid:
#                 # Process previous user's predictions
#                 if current_uid is not None:
#                     p, r = process_user_buffer(user_preds)
#                     sum_precision += p
#                     sum_recall += r
#                     n_users += 1
                
#                 # Start new user
#                 current_uid = uid
#                 user_preds = []
            
#             user_preds.append((est, true_r))
        
#         # Process last user
#         if current_uid is not None and user_preds:
#             p, r = process_user_buffer(user_preds)
#             sum_precision += p
#             sum_recall += r
#             n_users += 1
        
#         # Calculate averages
#         avg_precision = sum_precision / n_users if n_users > 0 else 0
#         avg_recall = sum_recall / n_users if n_users > 0 else 0
        
#         return {
#             "Algorithm": name,
#             "RMSE": round(rmse, 4),
#             "Precision@10": round(avg_precision, 4),
#             "Recall@10": round(avg_recall, 4),
#             "Fit Time (s)": round(fit_time, 4),
#             "Inference Time (s)": round(inference_time, 4)
#         }


# --- 5. Evaluator (TRUE Recommendation Evaluation) ---
class Evaluator:
    def evaluate_model(self, model, trainset, testset, name, k=10, threshold=7.0):
        """
        Evaluates recommendation system properly:
        1. Train on trainset
        2. For each user, recommend K items from ALL items they haven't seen in training
        3. Check if these recommendations appear in testset with high ratings
        
        Args:
            model: Recommendation model
            trainset: Surprise trainset object
            testset: List of (user, item, rating) tuples
            name: Model name
            k: Number of recommendations
            threshold: Rating threshold for "relevant" items
        """
        print(f"Training {name}...")
        
        # Training phase
        start_fit = time.time()
        model.fit(trainset)
        fit_time = time.time() - start_fit
        print(f"  Training completed in {fit_time:.2f}s")
        
        # Get all items and users
        all_items = set(trainset.all_items())
        all_item_ids = [trainset.to_raw_iid(i) for i in all_items]
        
        # Build ground truth from testset
        print(f"  Building ground truth...")
        test_user_items = defaultdict(dict)  # {user: {item: rating}}
        for uid, iid, rating in testset:
            test_user_items[uid][iid] = rating
        
        # Build training user-item map (to exclude already seen items)
        train_user_items = defaultdict(set)
        for uid, iid, rating in trainset.all_ratings():
            raw_uid = trainset.to_raw_uid(uid)
            raw_iid = trainset.to_raw_iid(iid)
            train_user_items[raw_uid].add(raw_iid)
        
        # Get users that appear in test set
        test_users = list(test_user_items.keys())
        
        print(f"  Generating recommendations for {len(test_users)} users...")
        
        # Generate recommendations and evaluate
        start_inference = time.time()
        
        sum_precision, sum_recall, n_users = 0, 0, 0
        sum_ndcg, sum_hit_rate = 0, 0
        
        for user_idx, user in enumerate(test_users):
            if (user_idx + 1) % 100 == 0:
                print(f"    Progress: {user_idx + 1}/{len(test_users)} users")
            
            # Get items user hasn't seen in training
            seen_items = train_user_items.get(user, set())
            candidate_items = [item for item in all_item_ids if item not in seen_items]
            
            # Predict scores for all unseen items
            predictions = []
            for item in candidate_items:
                try:
                    if hasattr(model, 'predict'):
                        pred = model.predict(user, item)
                        est = pred.est if hasattr(pred, 'est') else pred
                    else:
                        est = model.predict_score(user, item)
                    predictions.append((item, est))
                except:
                    # Skip items that can't be predicted
                    continue
            
            # Sort by predicted score and get top-k
            predictions.sort(key=lambda x: x[1], reverse=True)
            top_k_items = [item for item, score in predictions[:k]]
            
            # Get relevant items from test set (items with rating >= threshold)
            relevant_items = {item for item, rating in test_user_items[user].items() 
                            if rating >= threshold}
            
            # Calculate metrics
            if len(relevant_items) > 0 and len(top_k_items) > 0:
                # Hits: relevant items that appear in top-k recommendations
                hits = [item for item in top_k_items if item in relevant_items]
                n_hits = len(hits)
                
                # Precision@k = hits / k
                precision = n_hits / k
                
                # Recall@k = hits / total_relevant
                recall = n_hits / len(relevant_items)
                
                # Hit Rate: did we recommend at least one relevant item?
                hit_rate = 1.0 if n_hits > 0 else 0.0
                
                # NDCG@k
                dcg = sum([1.0 / np.log2(idx + 2) for idx, item in enumerate(top_k_items) 
                          if item in relevant_items])
                idcg = sum([1.0 / np.log2(idx + 2) for idx in range(min(k, len(relevant_items)))])
                ndcg = dcg / idcg if idcg > 0 else 0.0
                
                sum_precision += precision
                sum_recall += recall
                sum_hit_rate += hit_rate
                sum_ndcg += ndcg
                n_users += 1
        
        inference_time = time.time() - start_inference
        
        # Calculate RMSE on testset
        print(f"  Calculating RMSE...")
        rmse_sse = 0
        count = 0
        for uid, iid, true_r in testset:
            try:
                if hasattr(model, 'predict'):
                    pred = model.predict(uid, iid)
                    est = pred.est if hasattr(pred, 'est') else pred
                else:
                    est = model.predict_score(uid, iid)
                rmse_sse += (est - true_r) ** 2
                count += 1
            except:
                continue
        
        rmse = np.sqrt(rmse_sse / count) if count > 0 else 0
        
        # Calculate averages
        avg_precision = sum_precision / n_users if n_users > 0 else 0
        avg_recall = sum_recall / n_users if n_users > 0 else 0
        avg_hit_rate = sum_hit_rate / n_users if n_users > 0 else 0
        avg_ndcg = sum_ndcg / n_users if n_users > 0 else 0
        
        print(f"  Evaluation completed!\n")
        
        return {
            "Algorithm": name,
            "RMSE": round(rmse, 4),
            f"Precision@{k}": round(avg_precision, 4),
            f"Recall@{k}": round(avg_recall, 4),
            f"NDCG@{k}": round(avg_ndcg, 4),
            f"Hit Rate@{k}": round(avg_hit_rate, 4),
            "Fit Time (s)": round(fit_time, 4),
            "Inference Time (s)": round(inference_time, 4)
        }

In [11]:
# --- MAIN EXECUTION ---

# 1. Prepare Data
# Using 200,000 ratings sample. Ensure your environment has enough RAM.
loader = AnimeLoader(sample_size=20000) 

Locating Anime Dataset...
Loading Anime Metadata...
Loading top 20000 valid ratings (excluding 0s)...
Loaded 20000 ratings.
Final filtered content: 3637 anime titles.


In [12]:
trainset, testset = loader.get_surprise_train_test()
movies_df = loader.get_content_data()

Converting to Surprise Dataset...


In [13]:
# 2. Define Models
# REMOVED user_based/item_based to prevent memory crash on sparse data
models_to_test = [
    # CFRecommender('svd'),
    # CFRecommender('nmf'),
    ContentBasedRecommender(movies_df)
]

In [14]:
# 3. Evaluation
evaluator = Evaluator()
results = []

In [15]:
# print("Starting Evaluation Loop...", flush=True)
# for model in models_to_test:
#     name = getattr(model, 'algo_name', 'Content-Based')
#     # Note: evaluator now defaults to threshold=7.0 internally for Anime
#     res = evaluator.evaluate_model(model, trainset, testset, name, k=10,  # Top-10 recommendations
#         threshold=7.0 ) # Items with rating >= 7 are "relevant")
#     results.append(res)
#     print(f"Finished {name}", flush=True)

# #

# # 4. Run Individual Models
# print("before loop")
# for model in models_to_test:
#     print("in loop")
#     name = getattr(model, 'algo_name', 'Content-Based')
#     res = evaluator.evaluate_model(model, trainset, testset, name, k=10, threshold = 3)
#     results.append(res)
# 4. Run Individual Models
print("before loop")

for model in models_to_test:
    print("in loop")
    name = getattr(model, 'algo_name', 'Content-Based')
    
    # Run the evaluation
    res = evaluator.evaluate_model(model, trainset, testset, name, k=10, threshold = 3)
    
    # --- NEW: Print the result immediately ---
    print(f"--- Results for {name} ---")
    print(res)
    print("-" * 30) # Optional separator line for readability
    # -----------------------------------------

    results.append(res)

before loop
in loop
Training Content-Based...
Building TF-IDF matrix...
Building user profiles (Likes >= 6.0)...
  Training completed in 0.04s
  Building ground truth...
  Generating recommendations for 158 users...
    Progress: 100/158 users
  Calculating RMSE...
  Evaluation completed!

--- Results for Content-Based ---
{'Algorithm': 'Content-Based', 'RMSE': 2.2576, 'Precision@10': 0.0608, 'Recall@10': 0.0621, 'NDCG@10': 0.0665, 'Hit Rate@10': 0.4177, 'Fit Time (s)': 0.0416, 'Inference Time (s)': 364.5614}
------------------------------


In [16]:
#4. Hybrid Model
print("Building Hybrid Model...", flush=True)
cf_part = CFRecommender('svd')
cb_part = ContentBasedRecommender(movies_df)
hybrid = HybridRecommender(cf_part, cb_part, alpha=0.6)
res_hybrid = evaluator.evaluate_model(hybrid, trainset, testset, "Hybrid (SVD+Content)")
results.append(res_hybrid)

Building Hybrid Model...
Training Hybrid (SVD+Content)...
Building TF-IDF matrix...
Building user profiles (Likes >= 6.0)...
  Training completed in 0.30s
  Building ground truth...
  Generating recommendations for 158 users...
    Progress: 100/158 users
  Calculating RMSE...
  Evaluation completed!



In [17]:
# 5. Output
results_df = pd.DataFrame(results)
print("\n--- Final Evaluation Results ---")
print(results_df)
results_df.to_csv("anime_benchmark_results.csv", index=False)


--- Final Evaluation Results ---
              Algorithm    RMSE  Precision@10  Recall@10  NDCG@10  \
0         Content-Based  2.2576        0.0608     0.0621   0.0665   
1  Hybrid (SVD+Content)  1.3680        0.0801     0.0549   0.0949   

   Hit Rate@10  Fit Time (s)  Inference Time (s)  
0       0.4177        0.0416            364.5614  
1       0.4872        0.2964              8.0906  


In [ ]:
print()